# TTE map review

Maps TTE map tiles that DPS (`run_tte_map`) published in the MAAP DPS STAC catalog, through the MAAP
TiTiler. Three steps:

1. **What's published**: a table of every run so far, one row per collection and run label
   (`tte:tcc_label`, e.g. `hansen2010`, `metachmv2_2m`).
2. **Choose** the runs to map in `SHOW` (and optionally a region).
3. **Map**: each chosen run gets its own layers in the layer list (TTE classes, tree cover,
   abruptness, tile grid). **Hover** a tile for its metadata; **click** a pixel for its class, tree
   cover, abruptness and zone in every chosen run, side by side.

Run the cells yourself in a Hub kernel: JupyterLab only draws the map from output you created. The
map is also saved as an HTML file (open it in a browser, or use *Trust HTML* in JupyterLab).
Zoomed out to continental scale (below zoom 5), the TTE classes and tree cover come from a 1 km
**overview** of each run (one small COG; TiTiler is too slow to mosaic thousands of tiles at those
zooms). Step 3 builds or updates the overviews; zoomed in, the layers switch to the 30 m tiles.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'lib'))   # ../lib (run from notebooks/)
sys.path.insert(0, os.path.expanduser('~/code/tte/lib'))
import pandas as pd
import tte_review as tr
pd.set_option('display.max_colwidth', 60)

## 1. What's published

One row per run: the DPS STAC collection (MAAP names it `{user}__<algorithm>_<n>__{version}`; the
`_<n>` is added by MAAP), the run label, the tree cover it was made with, how many tiles, where they are
(`extent`: W, S, E, N in degrees), the DPS job tags and when the tiles were made.

In [ ]:
runs = tr.published_runs('montesano', 'run_tte_map')
runs

## 2. Choose

* `SHOW`: the runs to map, as `(collection, run)` pairs from the table. It starts with every run in
  the table; delete the ones you don't want. The map opens on the first run, with its classes switched
  on; the others are in the layer list (runs in other regions: pan there, or put that run first).
* `REGION`: `None` for all published tiles of those runs, or a lon/lat box `(W, S, E, N)`, e.g. one of
  `REGIONS`.
* `REFERENCE`: your original c2010 map (GEE `TTE_DATA_v201902v2`). It gets its own layers (TTE classes,
  tree cover), from its 1 km overview (`lib/tte_overview.py`), and at 30 m once it is published in the
  DPS STAC (`dps/DPS_tte_map.ipynb`, section 7); then its values are also in the click popup, above each
  run's. `None` leaves it out.

In [ ]:
SHOW = list(zip(runs.collection, runs.run))      # edit: keep the (collection, run) pairs to map
REGIONS = {'western Alaska': (-166.0, 63.5, -155.0, 69.0), 'Brooks Range': (-155.0, 66.5, -141.0, 69.0)}
REGION = None                                    # e.g. REGIONS['Brooks Range']
REFERENCE = tr.find_reference()                  # the original c2010 map (GEE TTE_DATA_v201902v2); None = leave it out
print('reference:', REFERENCE or 'none yet (tte_overview.build_reference_overview(); dps/DPS_tte_map.ipynb, section 7)')

for s in SHOW:
    print(s)

## 3. Map

`tr.build_overviews(SHOW)` builds the 1 km overview of each run that has none, or that has fewer tiles
than the run has now (about 0.3 s per tile with 12 threads, so a few minutes for thousands of tiles).

In [ ]:
tr.build_overviews(SHOW)
m = tr.review_map(SHOW, bbox=REGION, reference=REFERENCE)
OUT_HTML = os.path.expanduser('~/my-public-bucket/tte/review/TTE_review_'
                              + '_'.join(run for _, run in SHOW)[:120] + '.html')
os.makedirs(os.path.dirname(OUT_HTML), exist_ok=True)
m.save(OUT_HTML)
print('saved', OUT_HTML)
m